# AWS S3: Simple Storage Service

In [1]:
!pip install boto3

   ---------------------------------------- 0.0/14.6 MB ? eta -:--:--
   ------- -------------------------------- 2.6/14.6 MB 15.1 MB/s eta 0:00:01
   ---------------------------------- ----- 12.6/14.6 MB 32.9 MB/s eta 0:00:01
   ---------------------------------------  14.4/14.6 MB 33.5 MB/s eta 0:00:01
   ---------------------------------------- 14.6/14.6 MB 23.5 MB/s  0:00:00

   ---------------------------------------- 0/4 [jmespath]
   ---------------------------------------- 0/4 [jmespath]
   ---------- ----------------------------- 1/4 [botocore]
   ---------- ----------------------------- 1/4 [botocore]
   ---------- ----------------------------- 1/4 [botocore]
   ---------- ----------------------------- 1/4 [botocore]
   ---------- ----------------------------- 1/4 [botocore]
   ---------- ----------------------------- 1/4 [botocore]
   ---------- ----------------------------- 1/4 [botocore]
   ---------- ----------------------------- 1/4 [botocore]
   ---------- -------------


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


# Upload

In [2]:
import boto3

s3 = boto3.client(
    's3',
    aws_access_key_id='your-access-key', # ideally you want to put this into your environment
    aws_secret_access_key='your-secret-key',
    region_name='ap-south-1'
)

response = s3.list_buckets()

print("Buckets:")
for bucket in response['Buckets']:
    print(bucket['Name'])

Buckets:
loxfordacademy-ml-dataset-1973
my-first-bucket-ash322
temp-pawan-shop


In [4]:
s3.upload_file(
    'local_file.csv',
    'loxfordacademy-ml-dataset-1973', # This bucket should exist on AWS S3
    'local_file.csv'
)
print("File uploaded")

File uploaded


## Now check on AWS S3. The file should exist there.

# Download

In [5]:
import pandas as pd

obj = s3.get_object(Bucket='loxfordacademy-ml-dataset-1973',
                    Key='local_file.csv'
)

df = pd.read_csv(obj['Body'])
print(df.head())

       name  age
0       lox   23
1      ford   92
2  Rockwell   45


# ML: Create a ML Model, save it on AWS S3 and then use it to make inferencing

In [7]:
import joblib
import io
from sklearn.datasets import load_iris
from sklearn.ensemble import RandomForestClassifier

# STEP 1: TRAIN MODEL
iris = load_iris()
X, y = iris.data, iris.target

model = RandomForestClassifier()
model.fit(X, y)

print("Model trained")

Model trained


In [8]:
# STEP 2: SAVE MODEL DIRECTLY TO MEMORY AND UPLOAD

# AWS CONFIG
BUCKET_NAME = "loxfordacademy-ml-dataset-1973"
MODEL_KEY = "iris_model.pkl"

buffer = io.BytesIO()
joblib.dump(model, buffer)
buffer.seek(0)

# Upload directly to S3
s3.upload_fileobj(buffer, BUCKET_NAME, MODEL_KEY)

print("Model uploaded directly to S3")

Model uploaded directly to S3


In [9]:
# STEP 3: DOWNLOAD MODEL DIRECTLY TO MEMORY

download_buffer = io.BytesIO()
s3.download_fileobj(BUCKET_NAME, MODEL_KEY, download_buffer)

download_buffer.seek(0)

# Load model directly
loaded_model = joblib.load(download_buffer)

print("Model loaded from S3")

Model loaded from S3


In [10]:
# STEP 4: INFERENCE

prediction = loaded_model.predict([X[0]])

print("Prediction:", prediction)

Prediction: [0]
